# Notebook 00 â€” Setup & Orientation
## Sarvam AI Ã— Indic NLP: A *Speech and Language Processing* Companion

**Purpose:** Verify your environment, meet the Sarvam model family, and see why Indic NLP is fundamentally different from the English-centric world of the textbook.

---
**Prerequisites:** Run `pip install -r ../requirements.txt` and create a `.env` file in the repo root:
```
SARVAM_API_KEY=<your_key_here>
DEMO_MODE=True
```

### Install/import check, load .env, init client

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from utils.sarvam_helpers import load_client, reset_demo_counters
from data.sample_texts import SAMPLE_TEXTS, LANGUAGE_NAMES, ENGLISH_TRANSLATIONS
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

reset_demo_counters()
client = load_client()
print('âœ“ Environment OK â€” SarvamAI client ready')
print(f'âœ“ DEMO_MODE = {os.getenv("DEMO_MODE")}')

## Why Indic NLP is Hard â€” Background & Framing

Jurafsky & Martin (*Speech and Language Processing* opens with the ambition: *"Every question you ever wanted to ask about language."* But the textbook's running examples are almost entirely **English**. When we shift to the Indian subcontinent, every assumption breaks:

| Challenge | English assumption | Indic reality |
|-----------|-------------------|---------------|
| **Scripts** | 1 script (Latin) | 10+ scripts (Devanagari, Tamil, Telugu, Bengali, Gujaratiâ€¦) |
| **Morphology** | Relatively isolating | Agglutinative (Tamil, Telugu) or fusional (Hindi, Sanskrit) |
| **Word order** | SVO fixed | SOV default; Hindi allows scrambling |
| **Tokenization** | Whitespace works | Sandhi (Hindi), compound words (Tamil) break whitespace |
| **Code-mixing** | Rare | 10M+ speakers mix Hindi+English per sentence |
| **Resources** | 100B+ token corpora | Many languages: < 1B tokens |

**India has 22 scheduled languages** under the Constitution, written in at least 10 scripts. The top 8 languages by speaker count (Hindi, Bengali, Telugu, Tamil, Gujarati, Urdu, Kannada, Malayalam) together have **~1.2 billion speakers** â€” but most NLP research ignores them.

**Sarvam AI** was founded to fix this. Their model suite (Sarvam-M 24B, Bulbul TTS, Saaras STT, Mayura MT) is trained specifically on Indic language data from AI4Bharat corpora and proprietary sources.

### Language detection smoke test on all 4 sample texts

In [ ]:
reset_demo_counters()
from utils.sarvam_helpers import detect_language

results = []
for lang_code, text in SAMPLE_TEXTS.items():
    try:
        det = detect_language(client, text)
        results.append({
            'Expected': LANGUAGE_NAMES[lang_code],
            'Expected Code': lang_code,
            'Detected': det['language_code'],
            'Confidence': f"{det['confidence']:.2%}",
            'Text Preview': text[:40] + 'â€¦'
        })
    except Exception as e:
        results.append({'Expected': lang_code, 'Error': str(e)})

df = pd.DataFrame(results)
print('Language Detection Results:')
print(df.to_string(index=False))

### Unicode codepoint range bar chart (script diversity visualization)

This chart visualizes **why Indic NLP is fundamentally different from English NLP** at the encoding level.

Each bar represents a script's Unicode block — its starting codepoint and range width. Latin (English) occupies a tiny, low range, while India's 10+ scripts are spread across completely separate Unicode blocks.

**Why this matters:**

1. **India uses 10+ separate Unicode blocks** — unlike European languages that mostly share Latin. A single character-level model can't just learn one alphabet; it needs to handle widely scattered codepoint ranges.

2. **Tokenizers built for Latin break on Indic text** — BPE/WordPiece models trained on English see Indic characters as rare byte sequences, leading to over-fragmentation (we explore this in depth in notebook 01).

3. **Script detection is trivial via codepoint ranges** — you can identify which language a text is written in just by checking which Unicode block its characters fall into. This is essentially what the Language Detection API does under the hood.

4. **This diversity is the theme of the entire course** — India's linguistic richness isn't just about vocabulary differences; it's about fundamentally different writing systems, grammars, and morphologies. Every notebook that follows will show you where English-centric NLP assumptions break down in instructive ways.


In [ ]:
reset_demo_counters()

# Unicode block ranges — Indic scripts + global scripts for comparison
script_ranges = {
    # --- Global scripts (for comparison) ---
    'Latin (English)':    (0x0041, 0x007A),   # A-z
    'Arabic':             (0x0600, 0x06FF),
    'CJK (Chinese)':      (0x4E00, 0x9FFF),   # CJK Unified Ideographs
    'Hangul (Korean)':    (0xAC00, 0xD7AF),   # Hangul Syllables
    'Cyrillic (Russian)': (0x0400, 0x04FF),
    'Thai':               (0x0E00, 0x0E7F),
    # --- Indic scripts ---
    'Devanagari (Hindi)': (0x0900, 0x097F),
    'Bengali':            (0x0980, 0x09FF),
    'Gurmukhi (Punjabi)': (0x0A00, 0x0A7F),
    'Gujarati':           (0x0A80, 0x0AFF),
    'Odia':               (0x0B00, 0x0B7F),
    'Tamil':              (0x0B80, 0x0BFF),
    'Telugu':             (0x0C00, 0x0C7F),
    'Kannada':            (0x0C80, 0x0CFF),
    'Malayalam':          (0x0D00, 0x0D7F),
}

labels = list(script_ranges.keys())
starts = [v[0] for v in script_ranges.values()]
sizes  = [v[1] - v[0] for v in script_ranges.values()]

# Color: blue shades for global, warm palette for Indic
n_global = 6
n_indic = len(labels) - n_global
colors_global = plt.cm.Blues([0.3, 0.35, 0.4, 0.45, 0.5, 0.55])[:n_global]
colors_indic  = plt.cm.Set2.colors[:n_indic]
colors = list(colors_global) + list(colors_indic)

fig, ax = plt.subplots(figsize=(13, 7))
bars = ax.barh(labels, sizes, left=starts, color=colors, edgecolor="white", height=0.7)

# Add a visual separator between global and Indic
ax.axhline(y=n_global - 0.5, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(0x0020, n_global - 0.5, "  ── Indic scripts below ──", va="center",
        fontsize=9, color="gray", style="italic")

ax.set_xlabel("Unicode Codepoint")
ax.set_title("Unicode Block Ranges: Indic Scripts vs Global Scripts
"
             "(bar width = number of codepoints in block)", fontsize=13)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"U+{int(x):04X}"))
import seaborn as sns
sns.despine()
plt.tight_layout()
plt.savefig("../outputs/figures/00_unicode_ranges.png", dpi=120, bbox_inches="tight")
plt.show()
print("Figure saved to outputs/figures/00_unicode_ranges.png")


## Sarvam Model Family Overview

| Model | Type | Key Capability | API Endpoint |
|-------|------|---------------|--------------|
| **Sarvam-M 24B** | LLM (reasoning) | 22 Indic languages, math/code, structured output | `chat.completions` |
| **Mayura v1** | Neural MT | High-quality translation, 4 style modes | `text.translate` |
| **Sarvam-Translate v1** | Neural MT | Alternative MT model for comparison | `text.translate` |
| **Bulbul v3** | TTS | 11 Indian languages, temperature-based expressiveness | `text_to_speech.convert` |
| **Bulbul v2** | TTS (legacy) | Pitch/loudness control (older API style) | `text_to_speech.convert` |
| **Saaras v3** | ASR | ~19% WER, 4 transcription modes | `speech_to_text.transcribe` |
| **Sarvam Vision 3B** | Multimodal | Document intelligence, structured extraction | `vision` |

### Model Selection Guide
- **Translation task** â†’ Mayura v1 (formal/colloquial modes) or Sarvam-Translate v1
- **Text generation / reasoning in Hindi** â†’ Sarvam-M 24B (`reasoning_effort=low` for speed)
- **Voice applications** â†’ Bulbul v3 (temperature control) 
- **Transcription** â†’ Saaras v3 (verbatim, transliterate, or code-mixed modes)
- **Document OCR + extraction** â†’ Sarvam Vision 3B

### Textbook Connection
Each model maps to a chapter in the textbook:
- Mayura / Sarvam-Translate â†’ **Ch 13** (Machine Translation)
- Sarvam-M â†’ **Ch 10** (Transformers), **Ch 11** (Fine-tuning)
- Bulbul â†’ **Ch 16** (TTS / Speech Synthesis)
- Saaras â†’ **Ch 16** (ASR / Automatic Speech Recognition)

### Display sample texts with their English translations

In [ ]:
reset_demo_counters()

print('='*70)
print('CANONICAL SAMPLE TEXTS â€” used throughout all notebooks')
print('='*70)
for lang_code, text in SAMPLE_TEXTS.items():
    lang_name = LANGUAGE_NAMES[lang_code]
    translation = ENGLISH_TRANSLATIONS[lang_code]
    print(f'\n[{lang_name} / {lang_code}]')
    print(f'  {text}')
    print(f'  â†’ (EN) {translation}')
print('\nâœ“ Setup complete! Proceed to notebook 01_tokenization_morphology.ipynb')

---
## Krutrim AI â€” A Second Indic LLM Ecosystem

> **Note:** All Krutrim comparison cells require `KRUTRIM_CLOUD_API_KEY` set
> in `.env`. If you do not have a key, the cells will print a clear error and
> can be skipped â€” all Sarvam cells work independently.
>
> Get a key (with INR 10,000 free credits) at: cloud.olakrutrim.com

Krutrim is India's other major Indic AI platform, built by Ola (Bhavish Aggarwal).
Where Sarvam specialises in speech-first Indic APIs, Krutrim focuses on
**LLM reasoning** and **Indic embeddings** â€” making them complementary rather
than purely competing.

### Krutrim Model Family

| Model | Type | Key Capability | API |
|-------|------|---------------|-----|
| **Krutrim-spectre-v2** | LLM | Flagship chat/reasoning, 22 Indic languages | `chat.completions` |
| **Krutrim-2 (LLM-2)** | LLM (12B) | 128K context, Mistral-NeMo base, Indic fine-tuned | `chat.completions` |
| **Bhasantarit-mini** | Embeddings | 768-dim Indic semantic embeddings (Vyakyarth) | `embeddings` |
| **KrutrimTranslate** | Neural MT | 9 Indic languages <-> English | `language_labs.translate` |
| **Bhashik TTS** | Speech synthesis | Indic TTS via native SDK | `audio.speech.bhashik` |

### API Compatibility
Krutrim's `chat` and `embeddings` endpoints are **OpenAI-compatible** â€” the
same `openai` Python package works, just with a different `base_url`:
```python
from openai import OpenAI
client = OpenAI(
    api_key=os.environ["KRUTRIM_CLOUD_API_KEY"],
    base_url="https://cloud.olakrutrim.com/v1",
)
```
This means any code written for OpenAI's API runs on Krutrim with one line
changed â€” a significant ecosystem advantage for Indian developers.

### BharatBench: Krutrim-2 vs Competitors (published, 2024)

| Task | Krutrim-2 | Mistral-NeMo-12B | GPT-4 |
|------|-----------|-----------------|-------|
| IndicCOPA (commonsense) | **0.80** | 0.58 | ~0.78 |
| IndicSentiment | **0.95** | 0.71 | 0.90 |
| NER â€” Gujarati (error rate, lower is better) | **0.21** | 0.54 | 0.29 |
| NER â€” Kannada (error rate) | **0.20** | 0.31 | 0.25 |
| Hindi Retrieval (Vyakyarth) | **99.9%** | â€” | â€” |
| Bengali Retrieval | **98.7%** | â€” | â€” |

Krutrim-2 (12B parameters) matches or beats models 5-10x its size on most
Indic classification and retrieval tasks.


In [ ]:
# [DISABLED] Krutrim API key unavailable — uncomment if you have a key
#
# # N
# o
# t
# e
# b
# o
# o
# k
 # 0
# 0
 # â
# €
# ”
 # K
# r
# u
# t
# r
# i
# m
 # A
# P
# I
 # s
# m
# o
# k
# e
 # t
# e
# s
# t

# i
# m
# p
# o
# r
# t
 # s
# y
# s
# ,
 # o
# s

# s
# y
# s
# .
# p
# a
# t
# h
# .
# i
# n
# s
# e
# r
# t
# (
# 0
# ,
 # o
# s
# .
# p
# a
# t
# h
# .
# a
# b
# s
# p
# a
# t
# h
# (
# '
# .
# .
# '
# )
# )


# t
# r
# y
# :

    # f
# r
# o
# m
 # u
# t
# i
# l
# s
# .
# k
# r
# u
# t
# r
# i
# m
# _
# h
# e
# l
# p
# e
# r
# s
 # i
# m
# p
# o
# r
# t
 # l
# o
# a
# d
# _
# o
# p
# e
# n
# a
# i
# _
# c
# l
# i
# e
# n
# t
# ,
 # k
# r
# u
# t
# r
# i
# m
# _
# c
# h
# a
# t

    # k
# r
# u
# t
# r
# i
# m
 # =
 # l
# o
# a
# d
# _
# o
# p
# e
# n
# a
# i
# _
# c
# l
# i
# e
# n
# t
# (
# )


    # r
# e
# s
# p
# o
# n
# s
# e
 # =
 # k
# r
# u
# t
# r
# i
# m
# _
# c
# h
# a
# t
# (

        # k
# r
# u
# t
# r
# i
# m
# ,

        # m
# e
# s
# s
# a
# g
# e
# s
# =
# [
# {
# "
# r
# o
# l
# e
# "
# :
 # "
# u
# s
# e
# r
# "
# ,

                   # "
# c
# o
# n
# t
# e
# n
# t
# "
# :
 # "
# N
# a
# m
# e
 # t
# h
# e
 # 4
 # D
# r
# a
# v
# i
# d
# i
# a
# n
 # l
# a
# n
# g
# u
# a
# g
# e
# s
 # o
# f
 # I
# n
# d
# i
# a
 # i
# n
 # o
# n
# e
 # l
# i
# n
# e
# .
# "
# }
# ]
# ,

    # )

    # p
# r
# i
# n
# t
# (
# "
# K
# r
# u
# t
# r
# i
# m
# -
# s
# p
# e
# c
# t
# r
# e
# -
# v
# 2
 # s
# m
# o
# k
# e
 # t
# e
# s
# t
# :
# "
# )

    # p
# r
# i
# n
# t
# (
# f
# "
  # {
# r
# e
# s
# p
# o
# n
# s
# e
# .
# s
# t
# r
# i
# p
# (
# )
# }
# "
# )

    # p
# r
# i
# n
# t
# (
# )


    # # S
# i
# d
# e
# -
# b
# y
# -
# s
# i
# d
# e
# :
 # s
# a
# m
# e
 # q
# u
# e
# s
# t
# i
# o
# n
 # t
# o
 # S
# a
# r
# v
# a
# m
# -
# M

    # f
# r
# o
# m
 # u
# t
# i
# l
# s
# .
# s
# a
# r
# v
# a
# m
# _
# h
# e
# l
# p
# e
# r
# s
 # i
# m
# p
# o
# r
# t
 # l
# o
# a
# d
# _
# c
# l
# i
# e
# n
# t
# ,
 # c
# h
# a
# t
# _
# c
# o
# m
# p
# l
# e
# t
# e
# ,
 # r
# e
# s
# e
# t
# _
# d
# e
# m
# o
# _
# c
# o
# u
# n
# t
# e
# r
# s

    # r
# e
# s
# e
# t
# _
# d
# e
# m
# o
# _
# c
# o
# u
# n
# t
# e
# r
# s
# (
# )

    # s
# a
# r
# v
# a
# m
 # =
 # l
# o
# a
# d
# _
# c
# l
# i
# e
# n
# t
# (
# )

    # s
# a
# r
# v
# a
# m
# _
# r
# e
# s
# p
 # =
 # c
# h
# a
# t
# _
# c
# o
# m
# p
# l
# e
# t
# e
# (

        # s
# a
# r
# v
# a
# m
# ,

        # [
# {
# "
# r
# o
# l
# e
# "
# :
 # "
# u
# s
# e
# r
# "
# ,

          # "
# c
# o
# n
# t
# e
# n
# t
# "
# :
 # "
# N
# a
# m
# e
 # t
# h
# e
 # 4
 # D
# r
# a
# v
# i
# d
# i
# a
# n
 # l
# a
# n
# g
# u
# a
# g
# e
# s
 # o
# f
 # I
# n
# d
# i
# a
 # i
# n
 # o
# n
# e
 # l
# i
# n
# e
# .
# "
# }
# ]
# ,

    # )

    # i
# f
 # "
# <
# t
# h
# i
# n
# k
# >
# "
 # i
# n
 # s
# a
# r
# v
# a
# m
# _
# r
# e
# s
# p
# :

        # s
# a
# r
# v
# a
# m
# _
# r
# e
# s
# p
 # =
 # s
# a
# r
# v
# a
# m
# _
# r
# e
# s
# p
# .
# s
# p
# l
# i
# t
# (
# "
# <
# /
# t
# h
# i
# n
# k
# >
# "
# )
# [
# -
# 1
# ]
# .
# s
# t
# r
# i
# p
# (
# )


    # p
# r
# i
# n
# t
# (
# "
# S
# a
# r
# v
# a
# m
# -
# M
 # r
# e
# s
# p
# o
# n
# s
# e
# :
# "
# )

    # p
# r
# i
# n
# t
# (
# f
# "
  # {
# s
# a
# r
# v
# a
# m
# _
# r
# e
# s
# p
# .
# s
# t
# r
# i
# p
# (
# )
# }
# "
# )


# e
# x
# c
# e
# p
# t
 # E
# n
# v
# i
# r
# o
# n
# m
# e
# n
# t
# E
# r
# r
# o
# r
 # a
# s
 # e
# :

    # p
# r
# i
# n
# t
# (
# f
# "
# K
# r
# u
# t
# r
# i
# m
 # k
# e
# y
 # n
# o
# t
 # c
# o
# n
# f
# i
# g
# u
# r
# e
# d
# :
 # {
# e
# }
# "
# )

    # p
# r
# i
# n
# t
# (
# "
# S
# e
# t
 # K
# R
# U
# T
# R
# I
# M
# _
# C
# L
# O
# U
# D
# _
# A
# P
# I
# _
# K
# E
# Y
 # i
# n
 # .
# e
# n
# v
 # t
# o
 # e
# n
# a
# b
# l
# e
 # K
# r
# u
# t
# r
# i
# m
 # c
# o
# m
# p
# a
# r
# i
# s
# o
# n
 # c
# e
# l
# l
# s
# .
# "
# )

# e
# x
# c
# e
# p
# t
 # E
# x
# c
# e
# p
# t
# i
# o
# n
 # a
# s
 # e
# :

    # p
# r
# i
# n
# t
# (
# f
# "
# E
# r
# r
# o
# r
# :
 # {
# e
# }
# "
# )

